# C2 Function 1 – Review Usefulness Baseline

This notebook implements the first trained baseline for **Component 2: Review Usefulness Scoring**.

### Purpose
The goal of Function 1 is to learn a usefulness score from review text that can later support review ranking and fixed-budget evidence selection in Function 2.

### Development scope
- Dataset: cleaned Amazon Reviews 2023 smartphone development pilot
- Products: 10
- Cleaned reviews: 896
- Target: `log_helpful = log(1 + helpful_vote)`
- Split strategy: within-product chronological train / validation / test

### Workflow
1. Resolve project paths and load the cleaned pilot data.
2. Validate the usefulness target and stable review IDs.
3. Validate the chronological split.
4. Prepare leakage-safe text features.
5. Train simple baseline models.
6. Select the Ridge configuration using the validation set only.
7. Evaluate the selected model on the held-out test set.
8. Save predictions, metrics, configuration, model, and vectorizer.
9. Reload the saved artifacts and demonstrate inference.
10. Run basic inference and failure-handling checks.

> **Important:** Helpful votes are treated as a proxy supervision signal, not a perfect ground-truth measure of semantic usefulness.


In [1]:
from pathlib import Path
import json
import joblib

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, ndcg_score
from scipy.stats import spearmanr

print("Function 1 environment ready")


Function 1 environment ready


## 1. Resolve Project Paths

The notebook may be launched from the repository root, the `component_2` directory, or the `notebooks` directory.  
This cell resolves the Component 2 root first so that later file paths are reproducible and do not depend on fragile `../` assumptions.


In [2]:
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "component_2").exists():
    COMPONENT_ROOT = CURRENT_DIR / "component_2"
elif CURRENT_DIR.name == "component_2":
    COMPONENT_ROOT = CURRENT_DIR
elif CURRENT_DIR.name == "notebooks" and CURRENT_DIR.parent.name == "component_2":
    COMPONENT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError(
        f"Could not locate component_2 from current directory: {CURRENT_DIR}"
    )

PILOT_PATH = (
    COMPONENT_ROOT
    / "data"
    / "pilot"
    / "smartphone_pilot_cleaned.parquet"
)

SPLIT_PATH = (
    COMPONENT_ROOT
    / "results"
    / "eda"
    / "chronological_split_assignments.csv"
)

print("Current directory:", CURRENT_DIR)
print("Component root:", COMPONENT_ROOT)
print("\nPilot path:", PILOT_PATH)
print("Pilot exists:", PILOT_PATH.exists())
print("\nSplit path:", SPLIT_PATH)
print("Split exists:", SPLIT_PATH.exists())

if not PILOT_PATH.exists():
    raise FileNotFoundError(f"Pilot dataset not found: {PILOT_PATH}")

if not SPLIT_PATH.exists():
    raise FileNotFoundError(f"Chronological split file not found: {SPLIT_PATH}")


Current directory: d:\Research\Repository\J26-DS-349\component_2\notebooks
Component root: d:\Research\Repository\J26-DS-349\component_2

Pilot path: d:\Research\Repository\J26-DS-349\component_2\data\pilot\smartphone_pilot_cleaned.parquet
Pilot exists: True

Split path: d:\Research\Repository\J26-DS-349\component_2\results\eda\chronological_split_assignments.csv
Split exists: True


## 2. Load Cleaned Pilot Data and Validate the Target

The cleaned smartphone pilot is loaded here. Stable `review_id` values are checked because later Component 2 stages must join usefulness scores with C1 aspect/sentiment evidence and semantic representations reliably.

The target is:

`log_helpful = log(1 + helpful_vote)`

The log transformation reduces the extreme skew of raw helpful-vote counts while preserving their ordering.

### Proxy-target limitation
An observed helpful-vote count of zero does **not** necessarily mean that the review has no useful content. It may also reflect limited exposure or lack of voting activity. Therefore, helpful votes are used as a practical proxy supervision signal rather than perfect ground truth.


In [3]:
df = pd.read_parquet(PILOT_PATH)

print("Dataset loaded:", PILOT_PATH)
print("Rows:", len(df))
print("Products:", df["parent_asin"].nunique())
print("Unique review IDs:", df["review_id"].nunique())
print("Duplicate review IDs:", df["review_id"].duplicated().sum())

print("\nColumns:")
print(df.columns.tolist())

display(
    df[
        [
            "review_id",
            "parent_asin",
            "rating",
            "helpful_vote",
            "verified_purchase",
        ]
    ].head()
)


Dataset loaded: d:\Research\Repository\J26-DS-349\component_2\data\pilot\smartphone_pilot_cleaned.parquet
Rows: 896
Products: 10
Unique review IDs: 896
Duplicate review IDs: 0

Columns:
['review_id', 'rating', 'title', 'text', 'asin', 'parent_asin', 'timestamp', 'helpful_vote', 'verified_purchase', 'review_date', 'review_length_words', 'review_length_chars', 'age_days', 'raw_helpful', 'log_helpful', 'product_relative', 'age_adjusted']


,review_id,parent_asin,rating,helpful_vote,verified_purchase
0,26c3cd248dc3d01d,B0B8PDLMMS,5.0,1,True
1,cd97cd121c0fc4b7,B09HFW8GRB,5.0,0,True
2,293639dc0ec21eae,B09JJCXC2M,5.0,0,True
3,5a4857744877f56c,B09JJCXC2M,4.0,0,True
4,ca2d0d6e3ea776f8,B09PPWVQSB,2.0,0,True


In [4]:
if "log_helpful" not in df.columns:
    df["log_helpful"] = np.log1p(
        df["helpful_vote"].astype(float)
    )

assert df["review_id"].is_unique
assert df["log_helpful"].isna().sum() == 0

print("Target: log_helpful")
print(df["log_helpful"].describe())

print(
    "\nMissing target values:",
    df["log_helpful"].isna().sum(),
)


Target: log_helpful
count    896.000000
mean       0.517610
std        0.832361
min        0.000000
25%        0.000000
50%        0.000000
75%        0.693147
max        4.727388
Name: log_helpful, dtype: float64

Missing target values: 0


## 3. Validate the Chronological Split

The existing split assignments are reused rather than creating a new random split.

A chronological split is important here because review helpfulness can be affected by time and exposure. The test set therefore remains untouched during model selection.


In [5]:
split_df = pd.read_csv(SPLIT_PATH)

print("Split file rows:", len(split_df))
print(
    "Unique split review IDs:",
    split_df["review_id"].nunique()
)

print("\nSplit counts:")
print(split_df["split"].value_counts())


Split file rows: 896
Unique split review IDs: 896

Split counts:
split
train         625
test          137
validation    134
Name: count, dtype: int64


In [6]:
model_df = df.merge(
    split_df[
        [
            "review_id",
            "split",
        ]
    ],
    on="review_id",
    how="left",
    validate="one_to_one",
)

print("Merged rows:", len(model_df))
print(
    "Missing split assignments:",
    model_df["split"].isna().sum()
)

print("\nFinal split counts:")
print(model_df["split"].value_counts())

display(
    model_df[
        [
            "review_id",
            "parent_asin",
            "review_date",
            "split",
            "log_helpful",
        ]
    ].head()
)


Merged rows: 896
Missing split assignments: 0

Final split counts:
split
train         625
test          137
validation    134
Name: count, dtype: int64


,review_id,parent_asin,review_date,split,log_helpful
0,26c3cd248dc3d01d,B0B8PDLMMS,2021-09-08 23:51:02.923000+00:00,train,0.693147
1,cd97cd121c0fc4b7,B09HFW8GRB,2021-06-05 21:25:08.584000+00:00,train,0.000000
2,293639dc0ec21eae,B09JJCXC2M,2021-09-19 13:25:13.275000+00:00,train,0.000000
3,5a4857744877f56c,B09JJCXC2M,2020-05-19 20:45:45.585000+00:00,train,0.000000
4,ca2d0d6e3ea776f8,B09PPWVQSB,2022-04-18 21:49:41.193000+00:00,train,0.000000


In [7]:
assert len(model_df) == len(df)
assert model_df["review_id"].is_unique
assert model_df["split"].isna().sum() == 0
assert set(model_df["split"].unique()) == {
    "train",
    "validation",
    "test",
}

print("Chronological split validation passed.")


Chronological split validation passed.


## 4. Prepare Leakage-Safe Text Features

Function 1 uses only review text information as predictor input for this preliminary baseline.

`helpful_vote`, `log_helpful`, and other target-derived variables are explicitly excluded from model inputs. This prevents direct target leakage.

The model text is created by joining the review `title` and `text`.


In [8]:
LEAKAGE_COLUMNS = [
    "helpful_vote",
    "log_helpful",
    "raw_helpful",
    "product_relative",
    "age_adjusted",
]

print("Columns excluded from predictor inputs:")
for col in LEAKAGE_COLUMNS:
    if col in model_df.columns:
        print("-", col)


Columns excluded from predictor inputs:
- helpful_vote
- log_helpful
- raw_helpful
- product_relative
- age_adjusted


In [9]:
model_df["model_text"] = (
    model_df["title"]
    .fillna("")
    .astype(str)
    .str.strip()
    + " "
    + model_df["text"]
    .fillna("")
    .astype(str)
    .str.strip()
).str.strip()

print(
    "Empty model_text rows:",
    (model_df["model_text"] == "").sum()
)

display(
    model_df[
        [
            "review_id",
            "model_text",
            "log_helpful",
            "split",
        ]
    ].head()
)


Empty model_text rows: 0


,review_id,model_text,log_helpful,split
0,26c3cd248dc3d01d,Best phone yet I love this phone it has everyt...,0.693147,train
1,cd97cd121c0fc4b7,Value Stays charged,0.000000,train
2,293639dc0ec21eae,Very nice phone The renewed phone arrived in p...,0.000000,train
3,5a4857744877f56c,"Clean, unlocked and as described. Love the Siz...",0.000000,train
4,ca2d0d6e3ea776f8,Samsung Always Finds a Way to Ruin a Good Phon...,0.000000,train


In [10]:
train_df = model_df[
    model_df["split"] == "train"
].copy()

validation_df = model_df[
    model_df["split"] == "validation"
].copy()

test_df = model_df[
    model_df["split"] == "test"
].copy()

print("Train rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Test rows:", len(test_df))


Train rows: 625
Validation rows: 134
Test rows: 137


In [11]:
X_train = train_df["model_text"]
y_train = train_df["log_helpful"]

X_validation = validation_df["model_text"]
y_validation = validation_df["log_helpful"]

X_test = test_df["model_text"]
y_test = test_df["log_helpful"]

print("Target means:")
print("Train:", y_train.mean())
print("Validation:", y_validation.mean())
print("Test:", y_test.mean())


Target means:
Train: 0.5824578620958407
Validation: 0.4187326855300366
Test: 0.3184856699764303


### Split-distribution note

The mean target value decreases from the training split to the validation and test splits. Because the split is chronological, this may reflect temporal/exposure differences in observed helpful votes.

This is documented as a limitation and should not be interpreted automatically as a modelling error.


In [12]:
assert len(X_train) == len(y_train)
assert len(X_validation) == len(y_validation)
assert len(X_test) == len(y_test)

assert X_train.isna().sum() == 0
assert X_validation.isna().sum() == 0
assert X_test.isna().sum() == 0

assert (X_train.str.strip() != "").all()
assert (X_validation.str.strip() != "").all()
assert (X_test.str.strip() != "").all()

print("Leakage-safe feature preparation passed.")


Leakage-safe feature preparation passed.


## 5. Train Baseline Models

A simple model is intentionally used first so that the Function 1 behaviour is easy to inspect and reproduce.

Two baselines are evaluated:

- **Dummy Mean:** predicts the training-set mean for every review.
- **TF-IDF + Ridge Regression:** converts review text into TF-IDF features and learns a regularized regression model.

The TF-IDF vectorizer is fitted **only on the training split**.


In [13]:
def evaluate_predictions(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mae = mean_absolute_error(
        y_true,
        y_pred,
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred,
        )
    )

    # Spearman correlation is undefined when either input is constant.
    if (
        np.unique(y_true).size < 2
        or np.unique(y_pred).size < 2
    ):
        rho = np.nan
        p_value = np.nan
    else:
        rho, p_value = spearmanr(
            y_true,
            y_pred,
        )

    return {
        "MAE": float(mae),
        "RMSE": float(rmse),
        "Spearman": float(rho) if np.isfinite(rho) else np.nan,
        "Spearman_p": (
            float(p_value)
            if np.isfinite(p_value)
            else np.nan
        ),
    }


In [14]:
dummy_model = DummyRegressor(
    strategy="mean"
)

dummy_model.fit(
    np.zeros((len(y_train), 1)),
    y_train,
)

dummy_val_pred = dummy_model.predict(
    np.zeros((len(y_validation), 1))
)

dummy_metrics = evaluate_predictions(
    y_validation,
    dummy_val_pred,
)

print("Dummy validation metrics:")
print(dummy_metrics)


Dummy validation metrics:
{'MAE': 0.5926007040661087, 'RMSE': 0.737564000968537, 'Spearman': nan, 'Spearman_p': nan}


> The Dummy Mean model produces a constant prediction. Therefore, its Spearman correlation is mathematically undefined and is reported as `NaN` without raising a warning.


In [15]:
tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_features=10000,
    sublinear_tf=True,
)

X_train_tfidf = tfidf.fit_transform(
    X_train
)

X_validation_tfidf = tfidf.transform(
    X_validation
)

print(
    "Training TF-IDF shape:",
    X_train_tfidf.shape,
)

print(
    "Validation TF-IDF shape:",
    X_validation_tfidf.shape,
)

print(
    "Vocabulary size:",
    len(tfidf.vocabulary_),
)


Training TF-IDF shape: (625, 7206)
Validation TF-IDF shape: (134, 7206)
Vocabulary size: 7206


In [16]:
ridge_results = []

for alpha in [0.1, 1.0, 10.0]:
    model = Ridge(
        alpha=alpha
    )

    model.fit(
        X_train_tfidf,
        y_train,
    )

    val_pred = model.predict(
        X_validation_tfidf
    )

    metrics = evaluate_predictions(
        y_validation,
        val_pred,
    )

    ridge_results.append(
        {
            "model": "TFIDF_Ridge",
            "alpha": alpha,
            **metrics,
        }
    )

ridge_results_df = pd.DataFrame(
    ridge_results
)

display(ridge_results_df)


,model,alpha,MAE,RMSE,Spearman,Spearman_p
0,TFIDF_Ridge,0.1,0.582310,0.710793,0.349126,3.558321e-05
1,TFIDF_Ridge,1.0,0.536713,0.661993,0.383967,4.651768e-06
2,TFIDF_Ridge,10.0,0.543328,0.672643,0.410755,8.238698e-07


In [17]:
comparison_df = pd.concat(
    [
        pd.DataFrame(
            [
                {
                    "model": "DummyMean",
                    "alpha": np.nan,
                    **dummy_metrics,
                }
            ]
        ),
        ridge_results_df,
    ],
    ignore_index=True,
)

display(
    comparison_df.sort_values(
        "RMSE"
    )
)


,model,alpha,MAE,RMSE,Spearman,Spearman_p
2,TFIDF_Ridge,1.0,0.536713,0.661993,0.383967,4.651768e-06
3,TFIDF_Ridge,10.0,0.543328,0.672643,0.410755,8.238698e-07
1,TFIDF_Ridge,0.1,0.582310,0.710793,0.349126,3.558321e-05
0,DummyMean,NaN,0.592601,0.737564,NaN,NaN


## 6. Validation Ranking Metrics and Model Selection

Because the predicted usefulness score will later help **rank and select reviews**, regression error alone is not sufficient.

For each product, NDCG measures whether reviews with higher observed usefulness targets are placed nearer the top of the predicted ranking. Product-level scores are averaged across eligible product groups.

The validation set is used to compare candidate Ridge regularization values. The test set is not used for model selection.


In [18]:
def mean_product_ndcg(
    frame,
    predictions,
    k=10,
):
    temp = frame[
        [
            "parent_asin",
            "log_helpful",
        ]
    ].copy()

    temp["prediction"] = predictions

    scores = []

    for _, group in temp.groupby(
        "parent_asin"
    ):
        if len(group) < 2:
            continue

        y_true = group[
            "log_helpful"
        ].to_numpy().reshape(1, -1)

        y_score = group[
            "prediction"
        ].to_numpy().reshape(1, -1)

        score = ndcg_score(
            y_true,
            y_score,
            k=min(k, len(group)),
        )

        scores.append(score)

    if not scores:
        return np.nan

    return float(
        np.mean(scores)
    )


In [19]:
validation_results = []

for alpha in [0.1, 1.0, 10.0]:
    model = Ridge(
        alpha=alpha
    )

    model.fit(
        X_train_tfidf,
        y_train,
    )

    val_pred = model.predict(
        X_validation_tfidf
    )

    metrics = evaluate_predictions(
        y_validation,
        val_pred,
    )

    ndcg_5 = mean_product_ndcg(
        validation_df,
        val_pred,
        k=5,
    )

    ndcg_10 = mean_product_ndcg(
        validation_df,
        val_pred,
        k=10,
    )

    validation_results.append(
        {
            "model": "TFIDF_Ridge",
            "alpha": alpha,
            **metrics,
            "NDCG@5": ndcg_5,
            "NDCG@10": ndcg_10,
        }
    )

validation_results_df = pd.DataFrame(
    validation_results
)

display(validation_results_df)


,model,alpha,MAE,RMSE,Spearman,Spearman_p,NDCG@5,NDCG@10
0,TFIDF_Ridge,0.1,0.582310,0.710793,0.349126,3.558321e-05,0.592585,0.698966
1,TFIDF_Ridge,1.0,0.536713,0.661993,0.383967,4.651768e-06,0.656855,0.744745
2,TFIDF_Ridge,10.0,0.543328,0.672643,0.410755,8.238698e-07,0.706220,0.764598


### Validation-Based Model Selection

`TF-IDF + Ridge Regression` with **`alpha = 10.0`** is selected as the preliminary Function 1 model.

In the current validation run:

- `alpha = 1.0` achieved the lowest MAE and RMSE.
- `alpha = 10.0` achieved the strongest Spearman correlation, NDCG@5, and NDCG@10.
- The regression-error difference between `alpha = 1.0` and `alpha = 10.0` was relatively small.

Since the usefulness score will later support review ranking and fixed-budget evidence selection, **ranking quality is prioritized while retaining comparable regression error**.

The choice is based on the validation set only. The held-out test set is not used for model selection.


## 7. Evaluate Held-Out Test Performance

The selected configuration is now evaluated once on the untouched test split.

Reported metrics:

- **MAE / RMSE:** absolute prediction error on the log-helpfulness target.
- **Spearman:** monotonic association between predicted and observed usefulness ordering.
- **NDCG@5 / NDCG@10:** product-level ranking quality near the top of the review list.

These results are preliminary evidence from the smartphone development pilot, not final research conclusions.


In [20]:
SELECTED_ALPHA = 10.0

selected_model = Ridge(
    alpha=SELECTED_ALPHA
)

selected_model.fit(
    X_train_tfidf,
    y_train,
)

X_test_tfidf = tfidf.transform(
    X_test
)

test_pred = selected_model.predict(
    X_test_tfidf
)

test_metrics = evaluate_predictions(
    y_test,
    test_pred,
)

test_ndcg_5 = mean_product_ndcg(
    test_df,
    test_pred,
    k=5,
)

test_ndcg_10 = mean_product_ndcg(
    test_df,
    test_pred,
    k=10,
)

test_results = {
    "model": "TFIDF_Ridge",
    "alpha": SELECTED_ALPHA,
    **test_metrics,
    "NDCG@5": test_ndcg_5,
    "NDCG@10": test_ndcg_10,
}

display(
    pd.DataFrame([test_results])
)


,model,alpha,MAE,RMSE,Spearman,Spearman_p,NDCG@5,NDCG@10
0,TFIDF_Ridge,10.0,0.563167,0.665546,0.309733,0.00023,0.588248,0.651073


In [21]:
test_predictions_df = test_df[
    [
        "review_id",
        "parent_asin",
        "review_date",
        "helpful_vote",
        "log_helpful",
    ]
].copy()

test_predictions_df[
    "predicted_log_helpful"
] = test_pred

test_predictions_df[
    "prediction_error"
] = (
    test_predictions_df[
        "predicted_log_helpful"
    ]
    - test_predictions_df[
        "log_helpful"
    ]
)

display(
    test_predictions_df.head(10)
)


,review_id,parent_asin,review_date,helpful_vote,log_helpful,predicted_log_helpful,prediction_error
5,4b4e46ac390763fa,B0B8PDLMMS,2022-06-09 06:52:21.382000+00:00,0,0.000000,0.587237,0.587237
6,7ddd5e2ee3440476,B09JJCXC2M,2023-03-01 02:49:21.830000+00:00,17,2.890372,1.114625,-1.775747
7,198e0392108dbf2a,B09JJCXC2M,2022-11-15 18:45:05.184000+00:00,0,0.000000,0.637570,0.637570
18,862d861a45f42863,B0BMWTL2S6,2023-01-13 05:51:33.367000+00:00,1,0.693147,0.582362,-0.110785
20,104535faeb41b598,B07GYY6VTD,2019-11-24 02:54:00.623000+00:00,0,0.000000,0.435361,0.435361
34,4475e374c529fef7,B0BL9TKSBL,2023-02-16 21:54:48+00:00,1,0.693147,0.559803,-0.133344
45,023f31d43e68d437,B0BMWTL2S6,2023-02-20 04:40:07.086000+00:00,0,0.000000,0.479354,0.479354
46,a169628b15f3adae,B07GYY6VTD,2020-12-01 22:57:39.208000+00:00,0,0.000000,0.429806,0.429806
47,7f10dd98251adae7,B0BL9TKSBL,2023-02-21 13:20:38.655000+00:00,2,1.098612,0.470907,-0.627706
48,d22e67e42903da03,B08XRDVWQ2,2022-03-26 17:03:58.920000+00:00,0,0.000000,0.717618,0.717618


### Interpretation note

A review can have `helpful_vote = 0` and therefore `log_helpful = 0.0` while the model predicts a positive usefulness score.

This is not automatically an implementation error. The model learns patterns from review text, while observed helpful votes may also depend on exposure and voting activity. For this reason, helpful votes remain a **proxy supervision signal**.


In [22]:
dummy_test_pred = dummy_model.predict(
    np.zeros((len(y_test), 1))
)

dummy_test_metrics = evaluate_predictions(
    y_test,
    dummy_test_pred,
)

test_comparison_df = pd.DataFrame(
    [
        {
            "model": "DummyMean",
            "alpha": np.nan,
            **dummy_test_metrics,
            "NDCG@5": np.nan,
            "NDCG@10": np.nan,
        },
        test_results,
    ]
)

display(test_comparison_df)


,model,alpha,MAE,RMSE,Spearman,Spearman_p,NDCG@5,NDCG@10
0,DummyMean,NaN,0.594834,0.706376,NaN,NaN,NaN,NaN
1,TFIDF_Ridge,10.0,0.563167,0.665546,0.309733,0.00023,0.588248,0.651073


### Current PP1 pilot result

The latest verified held-out test run produced approximately:

| Metric | TF-IDF + Ridge |
|---|---:|
| MAE | 0.5632 |
| RMSE | 0.6655 |
| Spearman | 0.3097 |
| NDCG@5 | 0.5882 |
| NDCG@10 | 0.6511 |

For regression error, the Dummy Mean baseline produced approximately:

- MAE: 0.5948
- RMSE: 0.7064

Therefore, the trained model showed a modest improvement over the constant baseline on the current smartphone pilot. These values are **preliminary PP1 evidence**, not final cross-category performance claims.


## 8. Save Predictions, Metrics, Configuration, and Model Artifacts

The notebook saves reproducible evidence for later Component 2 stages.

Saved outputs include:

- held-out test predictions,
- test metrics,
- validation model comparison,
- test baseline comparison,
- model configuration,
- trained Ridge model,
- fitted TF-IDF vectorizer.

Binary model files may remain local if excluded by repository `.gitignore`; the notebook, configuration, and evaluation outputs can still provide version-controlled evidence.


In [23]:
USEFULNESS_RESULTS_DIR = (
    COMPONENT_ROOT
    / "results"
    / "usefulness"
)

MODEL_DIR = (
    COMPONENT_ROOT
    / "models"
)

USEFULNESS_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_PATH = (
    MODEL_DIR
    / "tfidf_ridge_usefulness.joblib"
)

VECTORIZER_PATH = (
    MODEL_DIR
    / "tfidf_vectorizer.joblib"
)

PREDICTIONS_PATH = (
    USEFULNESS_RESULTS_DIR
    / "test_predictions.csv"
)

METRICS_PATH = (
    USEFULNESS_RESULTS_DIR
    / "test_metrics.json"
)

CONFIG_PATH = (
    USEFULNESS_RESULTS_DIR
    / "model_config.json"
)

VALIDATION_PATH = (
    USEFULNESS_RESULTS_DIR
    / "validation_model_comparison.csv"
)

TEST_COMPARISON_PATH = (
    USEFULNESS_RESULTS_DIR
    / "test_baseline_comparison.csv"
)


In [24]:
joblib.dump(
    selected_model,
    MODEL_PATH,
)

joblib.dump(
    tfidf,
    VECTORIZER_PATH,
)

test_predictions_df.to_csv(
    PREDICTIONS_PATH,
    index=False,
)

validation_results_df.to_csv(
    VALIDATION_PATH,
    index=False,
)

test_comparison_df.to_csv(
    TEST_COMPARISON_PATH,
    index=False,
)

serializable_test_results = {
    key: (
        value.item()
        if isinstance(value, np.generic)
        else value
    )
    for key, value in test_results.items()
}

with open(
    METRICS_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        serializable_test_results,
        f,
        indent=2,
    )

model_config = {
    "model": "TF-IDF + Ridge Regression",
    "alpha": SELECTED_ALPHA,
    "target": "log1p(helpful_vote)",
    "text_input": "title + review text",
    "tfidf_ngram_range": [1, 2],
    "tfidf_min_df": 2,
    "tfidf_max_features": 10000,
    "tfidf_sublinear_tf": True,
    "split": (
        "within-product chronological "
        "train/validation/test"
    ),
    "train_rows": len(train_df),
    "validation_rows": len(validation_df),
    "test_rows": len(test_df),
    "model_selection": (
        "alpha selected using validation ranking "
        "performance with regression-error trade-off"
    ),
    "scope": "smartphone development pilot",
}

with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        model_config,
        f,
        indent=2,
    )

print("Model saved:", MODEL_PATH)
print("Vectorizer saved:", VECTORIZER_PATH)
print("Predictions saved:", PREDICTIONS_PATH)
print("Metrics saved:", METRICS_PATH)
print("Config saved:", CONFIG_PATH)
print("Validation comparison saved:", VALIDATION_PATH)
print("Test comparison saved:", TEST_COMPARISON_PATH)


Model saved: d:\Research\Repository\J26-DS-349\component_2\models\tfidf_ridge_usefulness.joblib
Vectorizer saved: d:\Research\Repository\J26-DS-349\component_2\models\tfidf_vectorizer.joblib
Predictions saved: d:\Research\Repository\J26-DS-349\component_2\results\usefulness\test_predictions.csv
Metrics saved: d:\Research\Repository\J26-DS-349\component_2\results\usefulness\test_metrics.json
Config saved: d:\Research\Repository\J26-DS-349\component_2\results\usefulness\model_config.json
Validation comparison saved: d:\Research\Repository\J26-DS-349\component_2\results\usefulness\validation_model_comparison.csv
Test comparison saved: d:\Research\Repository\J26-DS-349\component_2\results\usefulness\test_baseline_comparison.csv


## 9. Reload Saved Artifacts and Demonstrate Inference

This step verifies that Function 1 can operate after the notebook training session has finished.

A real held-out review is transformed with the saved TF-IDF vectorizer and scored by the saved Ridge model.

This demonstrates the Function 1 flow:

**Original review → text preparation → saved TF-IDF vectorizer → saved Ridge model → predicted usefulness score**


In [25]:
loaded_model = joblib.load(
    MODEL_PATH
)

loaded_vectorizer = joblib.load(
    VECTORIZER_PATH
)

sample_review = test_df.iloc[0]

sample_text = sample_review[
    "model_text"
]

sample_vector = loaded_vectorizer.transform(
    [sample_text]
)

sample_prediction = loaded_model.predict(
    sample_vector
)[0]

print("Review ID:", sample_review["review_id"])
print("Product:", sample_review["parent_asin"])
print("Observed helpful votes:", sample_review["helpful_vote"])

print(
    "Actual log helpful:",
    round(
        float(sample_review["log_helpful"]),
        4,
    ),
)

print(
    "Predicted log helpful:",
    round(
        float(sample_prediction),
        4,
    ),
)

print("\nReview text:")
print(sample_text[:500])


Review ID: 4b4e46ac390763fa
Product: B0B8PDLMMS
Observed helpful votes: 0
Actual log helpful: 0.0
Predicted log helpful: 0.5872

Review text:
I am writing the review on this phone. This phone is amazing. Stays cool and the charge lasts all day. In fact, I use maybe 30% in a day. The camera is amazing too.


## 10. Basic Inference and Failure-Handling Tests

These checks verify that the saved Function 1 model handles:

- a valid real review,
- empty text,
- invalid non-string input,
- previously unseen vocabulary,
- a missing model artifact.

These are lightweight engineering checks for the PP1 prototype. They do not replace broader final-research robustness testing.


In [26]:
def predict_usefulness(
    review_text,
    model,
    vectorizer,
):
    if not isinstance(review_text, str):
        raise TypeError(
            "review_text must be a string."
        )

    review_text = review_text.strip()

    if not review_text:
        raise ValueError(
            "review_text cannot be empty."
        )

    review_vector = vectorizer.transform(
        [review_text]
    )

    prediction = model.predict(
        review_vector
    )[0]

    return float(prediction)


In [27]:
valid_prediction = predict_usefulness(
    sample_text,
    loaded_model,
    loaded_vectorizer,
)

assert np.isfinite(valid_prediction)

print(
    "PASS - valid review inference:",
    round(valid_prediction, 4),
)


PASS - valid review inference: 0.5872


In [28]:
try:
    predict_usefulness(
        "   ",
        loaded_model,
        loaded_vectorizer,
    )

    print(
        "FAIL - empty review was accepted"
    )

except ValueError as error:
    print(
        "PASS - empty review rejected:",
        error,
    )


PASS - empty review rejected: review_text cannot be empty.


In [29]:
try:
    predict_usefulness(
        None,
        loaded_model,
        loaded_vectorizer,
    )

    print(
        "FAIL - invalid input was accepted"
    )

except TypeError as error:
    print(
        "PASS - invalid input rejected:",
        error,
    )


PASS - invalid input rejected: review_text must be a string.


In [30]:
unseen_text = (
    "xyzunseenword abcnewtoken "
    "battery quantumcamera example"
)

unseen_prediction = predict_usefulness(
    unseen_text,
    loaded_model,
    loaded_vectorizer,
)

assert np.isfinite(
    unseen_prediction
)

print(
    "PASS - unseen text handled:",
    round(unseen_prediction, 4),
)


PASS - unseen text handled: 0.3821


In [31]:
missing_model_path = (
    MODEL_DIR
    / "model_that_does_not_exist.joblib"
)

try:
    joblib.load(
        missing_model_path
    )

    print(
        "FAIL - missing model was loaded"
    )

except FileNotFoundError:
    print(
        "PASS - missing model file "
        "detected correctly"
    )


PASS - missing model file detected correctly


In [32]:
print(
    "Function 1 basic inference "
    "and failure-handling tests completed."
)


Function 1 basic inference and failure-handling tests completed.


## 11. Function 1 Completion Summary

### Completed in this notebook
- Stable review-ID validation
- Chronological split validation
- Leakage-safe text feature preparation
- Dummy Mean baseline
- TF-IDF + Ridge usefulness model
- Validation-based hyperparameter selection
- Held-out test evaluation
- MAE, RMSE, Spearman, NDCG@5, and NDCG@10 reporting
- Baseline comparison
- Prediction and metric persistence
- Model and vectorizer persistence
- Saved-model reload and real-review inference
- Basic input and artifact failure-handling checks

### Current limitations
- The current experiment uses the **smartphone development pilot**, not the final broader dataset.
- Helpful votes are a **proxy supervision signal** and are sparse.
- Only a simple TF-IDF + Ridge trained baseline has been implemented so far.
- Broader model comparison, second-category validation, and final statistical testing remain later research work.
- Function 2 still needs C1 aspect/sentiment integration, semantic review representations, supported-minority qualification, and constrained evidence selection.

### Status
**Function 1 core PP1 implementation is complete and demonstrable.**

Before committing the notebook, run **Restart Kernel → Run All** once and confirm that every cell completes successfully.
